# Template for MCP-Server as a Tool

_Exposing an MCP Server as a tool to an agent for it to connect to it in a uniform and agent-agnostic way._

An agent to connect to two MCP servers (math and weather) and select the right MCP tool based on the user query and returns a final response.

**Prerequisites**

Refer to the following instructions to run the MCP servers first.

1. Open two seperate **Anaconda Prompt** terminals. 

2. Activate environment `agentic_ai` in each terminal using command `conda activate agentic_ai`.

3. In one terminal, run _math_ server by executing command `python 4_tool_calling_mcp_math_server.py` that listens on port 8001.

4. In the other terminal, run _weather_ server by executing command `python 4_tool_calling_mcp_weather_server.py` that listens on port 8002.

In [ ]:
# First import class "MultiServerMCPClient" from module langchain_mcp_adapters.client, 
# class "SystemMessage", "HumanMessage" and "ToolMessage" from module "langchain_core.messages" and 
# class "ChatOllama" from module "langchain_ollama.chat_models"

# Code here
# ...

In [ ]:
# Set variables over the following information.
# Ollama endpoint is available at http://localhost:11434.
# Use model Qwen 3.5 2B over name "qwen3.5:2b"

# Code here
# ...

## Tools


In [145]:
# A single MCP client that connects to both servers.

mcp_client = MultiServerMCPClient(
    {
        "math": {
            "url": "http://localhost:8001/mcp",
            "transport": "streamable_http",
        },
        "weather": {
            "url": "http://localhost:8002/mcp",
            "transport": "streamable_http",
        },
    }
)

In [ ]:
# Get a list of (LangChain) tools from both the servers by calling function `get_tools()` on
# the MCP client instance and store the function output (the tools) in a variable for later use.

## Agent

In [ ]:
# Instruction to be used to set agent's behaviour

agent_instructions = """You are a simple assistant who can answer queries only on the following.
        1) Math expression evaluation
        2) Weather of a location (e.g. Mumbai, Bangalore, Delhi, Chennai and Delhi)

        Rules:
        - If a request is outside the above mentioned topics, politely decline and 
        inform user what you offer and let her re-enter the query.
        - For weather, extract location from the user queryand run the weather tool passing location as argument.
        - Do not call tools until all required fields are explicitly provided by the user.
        - For every new request, fetch parameter values from the user again. Do not reuse values from prior turns.

        For math expression evaluation:
        - Required field: expression
        - Extract math expression from the user query and call math(expression).

        For weather of a location:
        - Required fields: location
        - Extract location from the user query and call weather(location).
        - For now, weather tool is dummy and it returns same temperature for any location.

        For tools:
        - For tool calls, fill 'tool_calls' structure instead of JSON output in your response.

        For response
        - Prapare your response from tools messages, if exist.        
        """

In [ ]:
# Instantiate a chat client by calling constructor of class "ChatOllama" passing
# model name as argument to parameter "model",
# argument "False" to argument "reasoning", and
# Ollama endpoint as argument to parameter "base_url"

# Code here
# ...


# Bind the tools by passing the tools in a list as argument to the function
# "bind_tools" of the chat client instance.
# Store the returned tools-bound chat client in variable for later use.

# Code here
# ...

In [ ]:
# Maintain a list of all messages that flow across interactions
# Initialize the message list with first message of type SystemMessage.
# The constructor of the system message should be called with agent instructions
# as an argument to the parameter "content".

# Code here
# ...

In [ ]:
# User may provide one or multiple queries in single turn to be sent to client

query = "What is (2 + 3) * 4 - 5 / 2? And, what is the weather in Bangalore?"   # Two queries in one phrase

# Consider the above user query.
# Initialize a message of type "HumanMessage" passing the above
# query to the constructor of class and
# append that message instance to the message list.

# Code here
# ...

In [ ]:
# Now, pass the message list to the tools-bound chat client by calling
# its function "invoke" passing the message list into it.
# Store the returned message in a variable.

# Code here
# ...

In [ ]:
# Display the model-returned message to analyze element `tool_calls` to check
# if model has returned the correct tool name(s) with argument(s) for client to call those later

# Code here
# ...

In [ ]:
# Append last received message into message list

# Code here
# ...

In [ ]:
# Now, client loops over each tool name to call the tool manually for execution

for tool_call in client_message.tool_calls:
    print(f"Tool call (detailed): {tool_call}")
    tool_name = tool_call["name"].lower()               # Extracts tool name and converts to lower case
    selected_tool = next(tool for tool in TOOLS if tool.name == tool_name)
    print(f"Selected tool: {selected_tool}")
  
    raw_tool_message = await selected_tool.ainvoke(tool_call.get("args"))

    # Normalizes MCP tool output to text
    if isinstance(raw_tool_message, list):
        content = "\n".join(
            item.get("text", str(item)) if isinstance(item, dict) else str(item) for item in raw_tool_message
        )
    else:
        content = str(raw_tool_message)

    print(f"Arguments: {tool_call["args"]}, \nTool raw message: {raw_tool_message}, \nNormalized output: {content}\n")
    
    # Message received from tool gets added to message list
    messages.append(
        ToolMessage(
            content=content,
            tool_call_id=tool_call["id"],
            name=tool_call["name"],
        )
    )

In [ ]:
# Display the message list by passing the list into function `display` to check all messages
# including tool messages before preparing for final response

# Code here
# ...

In [ ]:
# To get the final message from the model, call function "invoke" of the tools-bound client
# passing message list into it and store the return message and print it as final message to the user

# Code here
# ...